# Week 1 — Setup & Data Loading

In [ ]:
!pip install numpy opencv-python matplotlib cupy-cuda12x

import os
from google.colab import drive

def mount_google_drive(drive_path="/content/drive"):
    if os.path.exists(drive_path):
        print("Google Drive is already mounted.")
    else:
        print("Mounting Google Drive...")
        drive.mount(drive_path)
        if os.path.exists(drive_path):
            print("Google Drive mounted successfully.")
        else:
            print("Failed to mount Google Drive. Please try again.")

mount_google_drive()

In [ ]:
import random
import json
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt
from typing import Tuple, List


def list_all_images(file_path):
    """List all image filenames in a directory."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Directory not found: {file_path}")
    images = [f for f in os.listdir(file_path) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if len(images) == 0:
        print("No images found in the directory.")
    return images


def cross_val_imgs(SEED, K, file_path, output_jsons):
    os.makedirs(output_jsons, exist_ok=True)
    all_images = list_all_images(file_path)
    if len(all_images) < K:
        raise ValueError(f"Need at least {K} images, found {len(all_images)}")

    random.seed(SEED)
    random.shuffle(all_images)

    n = len(all_images)
    fold_size = n // K

    for i in range(K):
        val_start = i * fold_size
        val_end = (i + 1) * fold_size if i < K - 1 else n
        val_files = all_images[val_start:val_end]
        train_files = all_images[:val_start] + all_images[val_end:]
        fold_data = {"train": train_files, "val": val_files}
        fold_path = os.path.join(output_jsons, f"fold_{i+1}.json")
        with open(fold_path, "w") as f:
            json.dump(fold_data, f, indent=4)
        print("Saved:", fold_path)

In [ ]:
class PatchShuffleDataLoader:
    def __init__(self, json_file: str, dataset_dir: str, batch_size: int = 8,
                 image_size: Tuple[int, int] = (32, 32), num_patches: int = 4,
                 shuffle: bool = True):
        """
        DataLoader with patch shuffling.
        Labels are permutation vectors of length num_patches^2.
        """
        self.dataset_dir = dataset_dir
        self.batch_size = batch_size
        self.image_size = image_size
        self.num_patches = num_patches
        self.shuffle = shuffle

        if not os.path.exists(json_file):
            output_jsons = os.path.dirname(json_file)
            print(f"Fold file not found, generating cross-validation splits in {output_jsons} ...")
            cross_val_imgs(SEED=2025, K=5, file_path=dataset_dir, output_jsons=output_jsons)

        with open(json_file, 'r') as f:
            data = json.load(f)
        self.train_files = data['train']
        self.val_files = data['val']

    def _shuffle_patches(self, image: np.ndarray, indices: np.ndarray = None) -> Tuple[np.ndarray, np.ndarray]:
        H, W = image.shape[0], image.shape[1]
        N = self.num_patches
        ph, pw = H // N, W // N

        if H % N != 0 or W % N != 0:
            raise ValueError(
                f"Image size ({H}, {W}) not divisible by num_patches={N}. "
                "Choose a different num_patches or resize/crop the image."
            )
        self.patch_size = (ph, pw)

        patches = []
        for i in range(N):
            for j in range(N):
                patches.append(image[i*ph:(i+1)*ph, j*pw:(j+1)*pw, :])
        patches = np.array(patches)

        num_total = N * N
        if indices is None:
            indices = np.random.permutation(num_total)
        else:
            indices = np.asarray(indices)
            if indices.shape != (num_total,):
                raise ValueError(f"indices must have shape ({num_total},), got {indices.shape}")
            if set(indices.tolist()) != set(range(num_total)):
                raise ValueError("indices must be a permutation of 0..N^2-1")

        shuffled_image = np.empty_like(image)
        label = np.empty(num_total, dtype=np.int64)

        for new_pos, original_id in enumerate(indices):
            out_i = new_pos // N
            out_j = new_pos % N
            shuffled_image[out_i*ph:(out_i+1)*ph, out_j*pw:(out_j+1)*pw, :] = patches[original_id]
            label[original_id] = new_pos

        return shuffled_image, label

    def _load_image(self, filepath: str) -> Tuple[np.ndarray, np.ndarray]:
        full_path = os.path.join(self.dataset_dir, filepath)
        image = cv.imread(full_path)
        if image is None:
            print(f"[Warning] Failed to load image: {full_path}. Skipping...")
            return None, None

        image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
        image = cv.resize(image, self.image_size)
        if image.ndim == 2:
            image = image[:, :, np.newaxis]

        image, labels = self._shuffle_patches(image)
        image = image.astype(np.float32) / 255.0
        image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
        return image, labels

    def _get_batches(self, file_list: List[str]):
        files = list(file_list)
        if self.shuffle:
            random.shuffle(files)
        for start in range(0, len(files), self.batch_size):
            batch_files = files[start:start + self.batch_size]
            X, Y = [], []
            for fname in batch_files:
                img, label = self._load_image(fname)
                if img is None:
                    continue
                X.append(img)
                Y.append(label)

            if len(X) > 0:
                yield np.stack(X, axis=0), np.stack(Y, axis=0)

    def train_batches(self):
        return self._get_batches(self.train_files)

    def val_batches(self):
        return self._get_batches(self.val_files)

    def display_training_image(self, image: np.ndarray):
        new_im = np.transpose(image, (1, 2, 0))
        new_im = np.clip(new_im * 255.0, 0, 255).astype(np.uint8)
        plt.imshow(new_im, cmap='gray')
        plt.axis("off")
        plt.show()

    def display_reconstructed_training_image(self, image: np.ndarray, label: np.ndarray):
        new_im = np.transpose(image, (1, 2, 0))
        new_im = np.clip(new_im * 255.0, 0, 255).astype(np.uint8)
        new_im, _ = self._shuffle_patches(new_im, label)
        plt.imshow(new_im, cmap='gray')
        plt.axis("off")
        plt.show()

# Week 2 — im2col, col2im & Convolution Layer

In [ ]:
import numpy as np
import cupy as cp


def im2col(X, kernel_size, stride=1, pad=0):
    """
    Convert image patches to columns for efficient convolution.

    Args:
        X: ndarray of shape (B, C, H, W)
        kernel_size: int or (KH, KW)
        stride: int
        pad: int

    Returns:
        H_out, W_out, cols of shape (C*KH*KW, B*H_out*W_out)
    """
    B, C, H, W = X.shape
    KH, KW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

    if pad > 0:
        X_p = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode="constant", constant_values=0)
    else:
        X_p = X

    H_p, W_p = X_p.shape[2], X_p.shape[3]
    H_out = (H_p - KH) // stride + 1
    W_out = (W_p - KW) // stride + 1

    patch_size = C * KH * KW
    cols = np.zeros((patch_size, B * H_out * W_out), dtype=X.dtype)

    col_idx = 0
    for b in range(B):
        for i in range(H_out):
            h0 = i * stride
            for j in range(W_out):
                w0 = j * stride
                cols[:, col_idx] = X_p[b, :, h0:h0+KH, w0:w0+KW].reshape(-1)
                col_idx += 1

    return H_out, W_out, cols


def col2im(cols, input_shape, kernel_size, stride=1, pad=0):
    """
    Convert columns back to image (inverse of im2col).
    """
    B, C, H, W = input_shape
    KH, KW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

    H_p = H + 2 * pad
    W_p = W + 2 * pad
    H_out = (H_p - KH) // stride + 1
    W_out = (W_p - KW) // stride + 1

    dX_p = np.zeros((B, C, H_p, W_p), dtype=cols.dtype)

    col_idx = 0
    for b in range(B):
        for i in range(H_out):
            h0 = i * stride
            for j in range(W_out):
                w0 = j * stride
                dX_p[b, :, h0:h0+KH, w0:w0+KW] += cols[:, col_idx].reshape(C, KH, KW)
                col_idx += 1

    if pad > 0:
        return dX_p[:, :, pad:pad+H, pad:pad+W]
    return dX_p

In [ ]:
class Layer:
    def __init__(self):
        self.input = None
        self.output = None

    def forward(self, X):
        raise NotImplementedError

    def backward(self, dY):
        raise NotImplementedError


class ConvLayer(Layer):
    def __init__(self, in_channels: int, out_channels: int,
                 kernel_size: int = 3, stride: int = 1, pad: int = 1):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.pad = pad

        # He initialization
        scale = np.sqrt(2.0 / (in_channels * kernel_size * kernel_size))
        self.W = np.random.randn(out_channels, in_channels, kernel_size, kernel_size) * scale
        self.b = np.zeros(out_channels)

    def forward(self, X):
        B, C, H, W = X.shape
        F = self.out_channels

        H_out, W_out, patches_col = im2col(X, self.kernel_size, self.stride, self.pad)
        W_col = self.W.reshape(F, -1)

        # GPU matmul
        out = cp.asnumpy(
            cp.matmul(cp.asarray(W_col), cp.asarray(patches_col))
            + cp.asarray(self.b[:, None])
        )

        out = out.reshape(F, B, H_out, W_out).transpose(1, 0, 2, 3)

        self.patches = patches_col
        self.H_in = H
        self.W_in = W
        return out

    def backward(self, dY):
        B, F, H_out, W_out = dY.shape
        C = self.in_channels
        H, W = self.H_in, self.W_in

        dY_reshaped = dY.transpose(1, 0, 2, 3).reshape(F, -1)

        # Gradient wrt weights (GPU)
        self.dW = cp.asnumpy(
            cp.matmul(cp.asarray(dY_reshaped), cp.asarray(self.patches.T))
        ).reshape(self.W.shape)

        # Gradient wrt bias (GPU)
        self.db = cp.asnumpy(cp.sum(cp.asarray(dY), axis=(0, 2, 3)))

        # Gradient wrt input (GPU)
        W_col = self.W.reshape(F, -1)
        dX_patches = cp.asnumpy(
            cp.matmul(cp.asarray(W_col.T), cp.asarray(dY_reshaped))
        )

        dX = col2im(dX_patches, (B, C, H, W), self.kernel_size, self.stride, self.pad)
        return dX

In [ ]:
import torch

def test_conv_layer():
    B, C, H, W = 2, 3, 5, 5
    F = 4
    kernel_size, stride, pad = 3, 1, 1

    X_np = np.random.randn(B, C, H, W).astype(np.float32)
    X_torch = torch.tensor(X_np, requires_grad=True)

    conv_np = ConvLayer(C, F, kernel_size=kernel_size, stride=stride, pad=pad)
    conv_torch = torch.nn.Conv2d(C, F, kernel_size, stride=stride, padding=pad, bias=True)

    with torch.no_grad():
        conv_torch.weight.copy_(torch.tensor(conv_np.W))
        conv_torch.bias.copy_(torch.tensor(conv_np.b))

    out_np = conv_np.forward(X_np)
    out_torch = conv_torch(X_torch)
    assert np.allclose(out_np, out_torch.detach().numpy(), atol=1e-5), "Forward mismatch"

    dY_np = np.random.randn(*out_np.shape).astype(np.float32)
    dX_np = conv_np.backward(dY_np)
    out_torch.backward(torch.tensor(dY_np))

    assert np.allclose(conv_np.dW, conv_torch.weight.grad.numpy(), atol=1e-5), "dW mismatch"
    assert np.allclose(conv_np.db, conv_torch.bias.grad.numpy(), atol=1e-5), "db mismatch"
    assert np.allclose(dX_np, X_torch.grad.numpy(), atol=1e-5), "dX mismatch"

    print("[INFO] ConvLayer tests passed!")

test_conv_layer()

# Week 3 — ReLU, MaxPool, Softmax & Cross-Entropy

In [ ]:
class ReLULayer(Layer):
    def forward(self, X):
        self.mask = (X > 0)
        return X * self.mask

    def backward(self, dY):
        return dY * self.mask


class MaxPoolLayer(Layer):
    def __init__(self, size=2, stride=2):
        super().__init__()
        self.size = size
        self.stride = stride

    def forward(self, X):
        self.X = X
        B, C, H, W = X.shape

        H_out, W_out, patches_col = im2col(X, self.size, self.stride)
        patches_reshaped = patches_col.reshape(C, self.size * self.size, -1)

        self.max_idx = np.argmax(patches_reshaped, axis=1)
        self.rows = np.arange(C)[:, None]
        self.cols = np.arange(B * H_out * W_out)

        out_flat = patches_reshaped[self.rows, self.max_idx, self.cols]
        out = out_flat.reshape(C, B, H_out, W_out).transpose(1, 0, 2, 3)

        self.H_in = H
        self.W_in = W
        return out

    def backward(self, dY):
        B, C, H_out, W_out = dY.shape
        H, W = self.H_in, self.W_in

        dpatches = np.zeros((C, self.size * self.size, B * H_out * W_out), dtype=dY.dtype)
        dpatches[self.rows, self.max_idx, self.cols] = \
            dY.transpose(1, 0, 2, 3).reshape(C, B * H_out * W_out)

        dpatches_col = dpatches.reshape(C * self.size * self.size, -1)
        dX = col2im(dpatches_col, (B, C, H, W), kernel_size=self.size, stride=self.stride)
        return dX


class ReshapeLayer(Layer):
    def __init__(self, new_shape):
        self.new_shape = new_shape
        self.input_shape = None

    def forward(self, X):
        self.input_shape = X.shape
        return X.reshape(X.shape[0], *self.new_shape)

    def backward(self, dY):
        return dY.reshape(self.input_shape)


class SoftmaxLayer(Layer):
    def __init__(self, axis: int = 1):
        super().__init__()
        self.axis = axis
        self.Y = None

    def forward(self, X: np.ndarray) -> np.ndarray:
        X_shift = X - np.max(X, axis=self.axis, keepdims=True)
        eX = np.exp(X_shift)
        self.Y = eX / np.sum(eX, axis=self.axis, keepdims=True)
        return self.Y

    def backward(self, dY: np.ndarray) -> np.ndarray:
        dot = np.sum(dY * self.Y, axis=self.axis, keepdims=True)
        return self.Y * (dY - dot)


class CrossEntropyLoss(Layer):
    def forward(self, X: np.ndarray, Y: np.ndarray, images: np.ndarray = None,
                eps_black: float = 0.05) -> float:
        """
        Args:
            X: Network output probabilities, shape (B, C, H, W).
            Y: Ground-truth permutation labels, shape (B, N^2).
            images: Original input images (B, 1, H_img, W_img). When provided,
                    black patches (all pixels < eps_black) are masked out of the loss.
            eps_black: Threshold below which a patch is considered black.
        """
        B, C, H, W = X.shape

        self.X_reshaped = np.transpose(X, (0, 2, 3, 1))  # (B, H, W, C)

        self.target_one_hot = np.zeros_like(self.X_reshaped, dtype=np.float32)
        self.target_one_hot[
            np.arange(B)[:, None, None],
            np.arange(H)[None, :, None],
            np.arange(W)[None, None, :],
            Y.reshape(B, H, W)
        ] = 1.0

        # Build per-patch mask: 1 for non-black, 0 for black
        if images is not None:
            _, _, H_img, W_img = images.shape
            ph, pw = H_img // H, W_img // W
            mask = np.ones((B, H, W), dtype=np.float32)
            for i in range(H):
                for j in range(W):
                    patch = images[:, 0, i*ph:(i+1)*ph, j*pw:(j+1)*pw]
                    is_black = np.all(patch < eps_black, axis=(1, 2))
                    mask[:, i, j] = (~is_black).astype(np.float32)
            self.mask = mask[:, :, :, np.newaxis]  # (B, H, W, 1)
        else:
            self.mask = np.ones((B, H, W, 1), dtype=np.float32)

        num_valid = max(self.mask.sum(), 1.0)

        loss = -np.sum(
            self.mask * self.target_one_hot * np.log(self.X_reshaped + 1e-12)
        ) / num_valid

        self._num_valid = num_valid
        return loss

    def backward(self) -> np.ndarray:
        dX_reshaped = -self.mask * self.target_one_hot / (self.X_reshaped + 1e-12)
        dX_reshaped /= self._num_valid
        return np.transpose(dX_reshaped, (0, 3, 1, 2))

In [ ]:
import torch

def test_relu_layer():
    B, C, H, W = 2, 3, 8, 8
    X_np = np.random.randn(B, C, H, W).astype(np.float32)
    X_torch = torch.tensor(X_np, requires_grad=True)

    relu_np = ReLULayer()
    out_np = relu_np.forward(X_np)
    out_torch = torch.nn.ReLU()(X_torch)
    assert np.allclose(out_np, out_torch.detach().numpy(), atol=1e-5), "ReLU forward mismatch"

    dY_np = np.random.randn(*out_np.shape).astype(np.float32)
    dX_np = relu_np.backward(dY_np)
    out_torch.backward(torch.tensor(dY_np))
    assert np.allclose(dX_np, X_torch.grad.numpy(), atol=1e-5), "ReLU dX mismatch"
    print("[INFO] ReLU layer tests passed!")


def test_maxpool_layer():
    B, C, H, W = 2, 3, 8, 8
    X_np = np.random.randn(B, C, H, W).astype(np.float32)
    X_torch = torch.tensor(X_np, requires_grad=True)

    maxpool_np = MaxPoolLayer(size=2, stride=2)
    out_np = maxpool_np.forward(X_np)
    out_torch = torch.nn.MaxPool2d(2, 2)(X_torch)
    assert np.allclose(out_np, out_torch.detach().numpy(), atol=1e-5), "MaxPool forward mismatch"

    dY_np = np.random.randn(*out_np.shape).astype(np.float32)
    dX_np = maxpool_np.backward(dY_np)
    out_torch.backward(torch.tensor(dY_np))
    assert np.allclose(dX_np, X_torch.grad.numpy(), atol=1e-5), "MaxPool dX mismatch"
    print("[INFO] MaxPool layer tests passed!")


def test_crossentropy_loss_layer():
    B, C, H, W = 2, 4, 3, 3

    logits = np.random.randn(B * H * W, C).astype(np.float32)
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    logprobs = np.log(probs)

    targets = np.random.randint(0, C, size=(B * H * W,))

    logprobs_torch = torch.tensor(logprobs, requires_grad=True)
    targets_torch = torch.tensor(targets, dtype=torch.long)

    probs_np = probs.reshape(B * H * W, 1, 1, C).transpose(0, 3, 1, 2)
    targets_np = targets.reshape(B * H * W, 1)

    loss_torch = torch.nn.NLLLoss()(logprobs_torch, targets_torch)

    ce_np = CrossEntropyLoss()
    loss_np = ce_np.forward(probs_np, targets_np)
    assert np.allclose(loss_np, loss_torch.item(), atol=1e-5), "CrossEntropy forward mismatch"

    loss_torch.backward()
    dX_torch_probs = logprobs_torch.grad.detach().numpy() / (probs + 1e-12)
    dX_np = ce_np.backward().transpose(0, 2, 3, 1).reshape(B * H * W, C)
    assert np.allclose(dX_np, dX_torch_probs, atol=1e-5), "CrossEntropy backward mismatch"

    print("[INFO] CrossEntropyLoss layer tests passed!")


test_relu_layer()
test_maxpool_layer()
test_crossentropy_loss_layer()

# Week 4 — Optimizers, Network & Utilities

In [ ]:
def _param_layers(layers):
    """Yield layers that have trainable weights."""
    for layer in layers:
        if hasattr(layer, 'W') and hasattr(layer, 'b'):
            yield layer


class SGD:
    def __init__(self, lr=1e-3):
        self.lr = lr

    def step(self, layers):
        for layer in _param_layers(layers):
            layer.W -= self.lr * layer.dW
            layer.b -= self.lr * layer.db


class Adam:
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0

    def step(self, layers):
        self.t += 1
        for layer in _param_layers(layers):
            if not hasattr(layer, 'mW'):
                layer.mW = np.zeros_like(layer.W)
                layer.vW = np.zeros_like(layer.W)
                layer.mb = np.zeros_like(layer.b)
                layer.vb = np.zeros_like(layer.b)

            layer.mW = self.beta1 * layer.mW + (1 - self.beta1) * layer.dW
            layer.vW = self.beta2 * layer.vW + (1 - self.beta2) * (layer.dW ** 2)
            mW_hat = layer.mW / (1 - self.beta1 ** self.t)
            vW_hat = layer.vW / (1 - self.beta2 ** self.t)
            layer.W -= self.lr * mW_hat / (np.sqrt(vW_hat) + self.eps)

            layer.mb = self.beta1 * layer.mb + (1 - self.beta1) * layer.db
            layer.vb = self.beta2 * layer.vb + (1 - self.beta2) * (layer.db ** 2)
            mb_hat = layer.mb / (1 - self.beta1 ** self.t)
            vb_hat = layer.vb / (1 - self.beta2 ** self.t)
            layer.b -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)


class Network:
    def __init__(self, layers, optimizer=None):
        self.layers = layers
        self.optimizer = optimizer or SGD()

    def forward(self, X):
        for layer in self.layers:
            X = layer.forward(X)
        return X

    def backward(self, dY):
        for layer in reversed(self.layers):
            dY = layer.backward(dY)
        return dY

    def step(self):
        self.optimizer.step(self.layers)

In [ ]:
import scipy.optimize


def assign_patches(probs):
    """
    Solve the patch assignment problem using predicted probabilities.

    Args:
        probs: np.ndarray of shape (C, H, W), where C = H * W,
               probs[i, h, w] = probability that patch (h, w)
               was originally at position i.

    Returns:
        np.ndarray of shape (C,) containing assigned original position for each patch.
    """
    C, H, W = probs.shape
    assert C == H * W, "Invalid shape for probabilities."

    cost = -probs.transpose(1, 2, 0).reshape(-1, C)
    row_ind, col_ind = scipy.optimize.linear_sum_assignment(cost)

    assignment = np.zeros(C, dtype=int)
    assignment[row_ind] = col_ind
    return assignment


def compute_reconstruction_accuracy(im, pred, gt, num_patches: int = 4, eps: float = 0.05):
    """
    Compute proportion of correctly reconstructed patches, ignoring black patches.
    """
    C, H, W = im.shape
    assert C == 1, "Only grayscale images supported."
    assert pred.shape == gt.shape
    assert len(pred.shape) == 1

    ph = H // num_patches
    pw = W // num_patches
    assert ph == pw, "Only square patches supported."

    im_2d = np.squeeze(im)
    num_correct = 0
    num_total = 0
    i = -1

    for y in range(0, H, ph):
        for x in range(0, W, pw):
            i += 1
            if np.all(im_2d[y:y+ph, x:x+pw] < eps):
                continue
            num_total += 1
            if pred[i] == gt[i]:
                num_correct += 1

    return num_correct / num_total


def compute_total_receptive_field(net):
    R = 1
    j = 1
    for layer in net.layers:
        if isinstance(layer, ConvLayer):
            k = layer.kernel_size
            s = layer.stride
            R += (k - 1) * j
            j = j * s
        elif isinstance(layer, MaxPoolLayer):
            k = layer.size
            s = layer.stride
            R += (k - 1) * j
            j = j * s
    return R

# Week 5 — Training

In [ ]:
mount_google_drive()

# ---- Configuration ----
dataset_dir = "/content/drive/MyDrive/Colab Notebooks/acv_exercises/datasets/acv_train_32x32"
json_file = "/content/drive/MyDrive/Colab Notebooks/acv_exercises/datasets/acv_train_32x32_cross_val/fold_1.json"

batch_size = 32
image_size = (32, 32)
num_patches = 4
num_epochs = 50
learning_rate = 1e-3

loader = PatchShuffleDataLoader(
    json_file,
    dataset_dir,
    batch_size=batch_size,
    image_size=image_size,
    num_patches=num_patches,
    shuffle=True
)

In [ ]:
# ---- Build network with Adam optimizer ----
# Switch to SGD(lr=learning_rate) if you want vanilla gradient descent
optimizer = Adam(lr=learning_rate)

net = Network([
    ConvLayer(in_channels=1, out_channels=32, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    ConvLayer(in_channels=32, out_channels=32, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    MaxPoolLayer(size=2, stride=2),   # 32x32 -> 16x16

    ConvLayer(in_channels=32, out_channels=64, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    ConvLayer(in_channels=64, out_channels=64, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    MaxPoolLayer(size=2, stride=2),   # 16x16 -> 8x8

    ConvLayer(in_channels=64, out_channels=128, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    ConvLayer(in_channels=128, out_channels=128, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    MaxPoolLayer(size=2, stride=2),   # 8x8 -> 4x4

    ConvLayer(in_channels=128, out_channels=64, kernel_size=3, stride=1, pad=1),
    ReLULayer(),

    ConvLayer(in_channels=64, out_channels=64, kernel_size=3, stride=1, pad=1),

    ConvLayer(in_channels=64, out_channels=16, kernel_size=1, stride=1, pad=0),

    SoftmaxLayer(axis=1),
], optimizer=optimizer)

print(f"Receptive field: {compute_total_receptive_field(net)} pixels")

In [ ]:
import pickle

def save_checkpoint(net, epoch, checkpoint_dir="checkpoints"):
    os.makedirs(checkpoint_dir, exist_ok=True)
    path = os.path.join(checkpoint_dir, f"epoch_{epoch}.pkl")
    with open(path, "wb") as f:
        pickle.dump(net, f)
    return path

# Google Drive checkpoint directory
drive_checkpoint_dir = "/content/drive/MyDrive/Colab Notebooks/acv_exercises/checkpoints"

# ---- Training loop ----
for epoch in range(num_epochs):
    epoch_loss = 0.0
    iter_train = 0

    for X_batch, Y_batch in loader.train_batches():
        iter_train += 1

        out_batch = net.forward(X_batch)

        loss_layer = CrossEntropyLoss()
        loss = loss_layer.forward(out_batch, Y_batch, images=X_batch)
        epoch_loss += loss

        dX = loss_layer.backward()
        net.backward(dX)
        net.step()

    avg_loss = epoch_loss / max(iter_train, 1)

    # ---- Validation accuracy ----
    total_acc = 0.0
    num_val = 0

    for X_val, Y_val in loader.val_batches():
        out_val = net.forward(X_val)
        B = X_val.shape[0]

        for i in range(B):
            pred = assign_patches(out_val[i])
            acc = compute_reconstruction_accuracy(X_val[i], pred, Y_val[i], num_patches=num_patches)
            total_acc += acc
            num_val += 1

    val_acc = total_acc / max(num_val, 1)

    # Save locally (always works even if internet drops)
    local_path = save_checkpoint(net, epoch)

    # Save to Google Drive (skip if connection lost)
    drive_path = None
    try:
        drive_path = save_checkpoint(net, epoch, checkpoint_dir=drive_checkpoint_dir)
    except Exception as e:
        print(f"[Warning] Drive save failed: {e}. Local checkpoint saved at {local_path}")

    saved_msg = f"Drive: {drive_path}" if drive_path else f"Local only: {local_path}"
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | {saved_msg}")